# Dam removal and downstream water quality: what the statistical models tell us

*A guided summary of the modeling work, for interpretation and for the capstone status update. Read-only: it loads the outputs of the latest corrected analysis (8 November 2026) and fits nothing.*

**How to read this notebook.** Six short sections and a summary. The text gives the interpretation; the tables and the two figures are loaded from the result files, so the numbers always match the latest run. Words used throughout:

- **Detected:** the 95% confidence interval excludes zero.
- **No meaningful change:** the *whole* interval lies inside the practical-significance threshold.
- **Inconclusive:** neither of the above; the data can't distinguish a meaningful change from no change.
- **Practical-significance threshold:** the smallest change we chose to call worth caring about (e.g. ±0.5 °C). These are **analyst-selected, provisional thresholds, not regulatory or environmental standards**: informed by aquatic-life research, water-quality criteria and instrument accuracy, but not fixed by them. Basis, caveats and sources: `docs/practical_significance_thresholds.md`.

In [ ]:
# Setup: paths, loaders and labels. Nothing is fitted or written.
import os
import sys
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "etl").is_dir() and (p / "config").is_dir())
sys.path.insert(0, str(ROOT))
from analysis import mixed_model as mm
from analysis import sensitivity_compare as sc
from analysis import stage4_report as rep

CASE_STUDY = "all_removals"
DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
MODEL_DIR = Path(os.environ.get("DAM_REMOVAL_MODEL_DIR", mm.output_dir_for(CASE_STUDY, DATA_ROOT)))
CORRECTED_ON = datetime(2026, 11, 8, 17, 40)   # corrected tables built then; earlier model outputs are outdated

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titlesize": 10, "axes.labelsize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8})

SERIES = ["water_temp", "dissolved_oxygen", "ph", "specific_conductance", "turbidity",
          "phosphorus (Total)", "orthophosphate (Dissolved)", "tds", "ssc", "tss"]
NAMES = {"water_temp": "Water temperature", "dissolved_oxygen": "Dissolved oxygen", "ph": "pH",
         "specific_conductance": "Specific conductance", "turbidity": "Turbidity",
         "phosphorus (Total)": "Total phosphorus", "orthophosphate (Dissolved)": "Orthophosphate",
         "tds": "Total dissolved solids", "ssc": "Suspended sediment (SSC)", "tss": "Total suspended solids"}
DESIGN = {"before_after_distance": "BA", "baci_distance": "BACI"}
COLORS = {"detected": "#c0392b", "no meaningful change": "#27ae60", "inconclusive": "#7f8c8d"}


def read_csv(path):
    return pd.read_csv(path) if path.exists() and path.stat().st_size > 1 else pd.DataFrame()


def run_dir(folder, run):
    # Output folder of a distance-model run, preferring the reported random-effects choice (auto).
    for re_mode in ("auto", "slopes", "intercepts"):
        d = MODEL_DIR / folder / f"{run}__{re_mode}"
        if (d / "results.csv").exists():
            return d
    return None


def load(folder, run, name="results.csv"):
    d = run_dir(folder, run)
    return read_csv(d / name) if d else pd.DataFrame()


# Authoritative inputs (latest corrected run)
primary = load("stage3", "primary")                       # BA and BACI distance models (primary)
primary_at = load("stage3", "primary", "effects_at.csv")  # effects at 1-30 km
mixed = read_csv(MODEL_DIR / "results.csv")               # original 10 km mixed models (corroboration)
time_res = load("stage5", "time")
time_at = load("stage5", "time", "effects_at.csv")
time_cmp = load("stage5", "time", "time_comparison.csv")
time_slopes = load("stage5", "time_ml_slopes", "time_comparison.csv")
lodo = read_csv(MODEL_DIR / "stage4" / "influence" / "lodo_summary.csv")
boot = read_csv(MODEL_DIR / "stage4" / "influence" / "bootstrap_summary.csv")
sens = {label: (load(f"sensitivity_{v}", "primary"),
                read_csv(MODEL_DIR / f"sensitivity_{v}" / "mixed_model" / "results.csv"))
        for label, v in {"3 / 1": "min1yr", "3 / 1, ≥2 months": "min1yr_2mo"}.items()}
known = load("sensitivity_known_controls", "known_controls")
known_mixed = read_csv(MODEL_DIR / "sensitivity_known_controls" / "mixed_model" / "results.csv")

# Freshness check: every input must come from the corrected run.
inputs = {"primary distance models": run_dir("stage3", "primary"), "mixed models": MODEL_DIR,
          "time models": run_dir("stage5", "time"), "influence": MODEL_DIR / "stage4" / "influence",
          "known-controls sensitivity": run_dir("sensitivity_known_controls", "known_controls")}
rows = []
for name, d in inputs.items():
    f = None if d is None else next((d / n for n in ("results.csv", "lodo_summary.csv") if (d / n).exists()), None)
    when = datetime.fromtimestamp(f.stat().st_mtime) if f else None
    rows.append({"input": name, "file": str(f.relative_to(MODEL_DIR)) if f else "MISSING",
                 "written": when.strftime("%Y-%m-%d %H:%M") if when else "",
                 "current": bool(when and when >= CORRECTED_ON)})
fresh = pd.DataFrame(rows)
print(f"Model outputs: {MODEL_DIR}")
display(fresh)
if not fresh["current"].all():
    print("WARNING: some inputs are missing or predate the 8 Nov 2026 corrections; numbers below may be outdated.")

In [ ]:
# Helpers used by the tables and figures.
SESOI = rep.SESOI


def natural(value, log_scale):
    # Model units -> readable units: log10 effects become % change.
    return (10 ** value - 1) * 100 if log_scale else value


def unit_of(series, log_scale):
    if log_scale:
        return "%"
    return {"water_temp": "°C", "dissolved_oxygen": "mg/L", "ph": "pH units"}.get(series, "")


def threshold_text(series):
    return rep.sesoi_label(series)


def fmt(est, lo, hi, series, log_scale):
    u = unit_of(series, log_scale)
    e, l, h = (natural(v, log_scale) for v in (est, lo, hi))
    d = 0 if log_scale else (3 if series == "ph" else 2)
    sign = lambda v: f"{v:+.{d}f}"
    return f"{sign(e)} [{sign(l)}, {sign(h)}] {u}".strip()


def verdict(est, lo, hi, series):
    return rep.classify(est, lo, hi, SESOI.get(series))

## 1. Research question and approach

**Question.** When a dam is removed, does water quality in the river downstream change, and if so, how much, how far downstream, and for how long?

**Data.** Dam removals across the U.S. (American Rivers database) linked to public water-quality monitoring stations (Water Quality Portal) along the river network. Each removal is studied over the **5 years before vs the 5 years after** removal (the removal year itself is left out).

**Measures.** Water temperature, dissolved oxygen (DO), pH, specific conductance (a proxy for dissolved salts), turbidity (cloudiness), total phosphorus and orthophosphate (nutrients), total dissolved solids (TDS), and suspended sediment (SSC, TSS).

**Why models, not national averages?** Comparing the average of all "after" samples with all "before" samples would mix up many things: different dams and rivers, samples taken in different seasons or weather, and many samples from a few heavily monitored sites. The models compare each site **with itself**, adjust for season and weather, and treat **dams** (not individual samples) as the independent units of evidence.

**Two study designs**

| | Before-after (**BA**) | Before-after-control-impact (**BACI**) |
|---|---|---|
| Compares | downstream stations after vs before removal | the downstream change vs the change at **upstream control** stations of the same dam over the same years |
| Answers | *Did water quality downstream change?* | *Did it change more than at comparable unaffected sites?* |
| Strength | many dams (≈ 100–280 per measure) | removes changes shared with the upstream river (weather years, regional trends), but few dams have upstream controls (≈ 20–30) |

BA provides evidence of **observed changes**; BACI is the **stronger comparative design**. Neither, on its own, establishes that the removal **caused** a change: these are observational data.

## 2. Understanding the models

All models share the same building blocks.

- **The response *y*** is one value per station per day (the median of that day's samples). Skewed measures (turbidity, nutrients, solids, conductance) are on a log10 scale, so their effects read as **% changes**.
- **Fixed effects** are the quantities we want to estimate, the same for every dam (e.g. the average change after removal).
- **Random effects** let each **dam** and each **station** have its own baseline level, and, where the data support it, each dam its own response. They are how the model handles **clustering**: hundreds of samples from one site are not hundreds of independent pieces of evidence. Ignoring this makes intervals far too narrow; with it, the uncertainty reflects **how much dams disagree**.
- **95% confidence intervals** use the number of dams (not samples) as the sample size. A wide interval means *uncertain*, not *small*.
- **Detectable vs important.** An effect can be statistically detectable but smaller than the practical-significance threshold; both facts are reported.

### A. Before-after (BA) model

$$
y = \beta_0 + \beta_1\,\text{post} + \text{month} + \gamma_1\,\text{air temp} + \gamma_2\,\text{last month's rain} + u_{\text{dam}} + b_{\text{dam}}\,\text{post} + v_{\text{station}} + \varepsilon
$$

| Term | Meaning |
|---|---|
| **post** | 1 for samples after removal, 0 before |
| **β₁** | **the estimated average change after removal**, given the season and weather adjustments |
| month | season adjustment (each calendar month its own level) |
| air temp, last month's rain | weather adjustment (standardized) |
| *u*<sub>dam</sub>, *v*<sub>station</sub> | each dam's and station's own baseline (random intercepts) |
| *b*<sub>dam</sub> | each dam's own deviation from the average change (random slope) |

*In words:* at the same downstream sites, in the same month and similar weather, how different is water quality after removal than before, on average across dams?

### B. BACI model

$$
y = \beta_0 + \beta_1\,\text{post} + \beta_2\,\text{down} + \beta_3\,(\text{post}\times\text{down}) + \text{month} + \text{weather} + u_{\text{dam}} + \text{dam slopes} + v_{\text{station}} + \varepsilon
$$

| Term | Meaning |
|---|---|
| **down** | 1 for downstream stations, 0 for upstream controls |
| β₁ (**post**) | the change after removal at the **upstream controls** (what happened anyway: weather years, regional trends) |
| β₂ (**down**) | the usual difference between downstream and upstream, before removal |
| **β₃ (post × down)** | **the BACI effect** |

**The interaction is the key term.** Downstream, the change after removal is β₁ + β₃; upstream it is β₁. So **β₃ = (downstream change) − (upstream change)**: how much *more* (or less) the downstream river changed than the upstream river did over the same years.

*Analogy:* a new teaching method in one class. Its scores rise 8 points, but a comparison class without the method rose 5 points over the same term (an easier exam, a better year). The method's estimated effect is the **difference**, 3 points, not 8.

### C. Distance models (the primary specification)

Removals may matter most near the dam, so the effect is allowed to vary with distance downstream (stations up to 30 km). With $d = \ln(1+\text{km}) - \ln(6)$, which is **0 at 5 km**:

$$
\text{BACI: } y = \dots + \beta_3\,(\text{post}\times\text{down}) + \beta_5\,(\text{post}\times\text{down}\times d) + \beta_7\,(\text{post}\times\text{down}\times r) + \dots
\qquad \text{effect at distance } d = \beta_3 + \beta_5\,d
$$

$$
\text{BA: } y = \beta_0 + \beta_1\,\text{post} + \beta_2\,d + \beta_3\,(\text{post}\times d) + \beta_4\,r + \beta_5\,(\text{post}\times r) + \dots \qquad \text{effect} = \beta_1 + \beta_3\,d
$$

(the BACI model also keeps down × *d* and down × *r* for baseline differences.)

- **The effect at 5 km** (β₃ for BACI, β₁ for BA) is the headline number.
- **The distance slope** (β₅, β₃) says how the effect changes as you move downstream: a slope of the **opposite sign** to the effect means the effect **fades** farther away; the same sign means it **grows**.
- ***r*** is a dilution proxy: how much more river has joined by the station (log drainage-area ratio, centered). It adjusts the effect so distance isn't confused with dilution.
- Random effects are chosen per measure: dam and station intercepts, plus dam-specific effects (random slopes) where a likelihood-ratio test supports them.

### D. Time-varying models

The single *post* switch is replaced by intervals after removal: **year 1, years 2–3, years 4–5** (vs the 5 years before). Each interval gets its own effect, so we can see whether a response appears quickly, fades, or builds, instead of assuming one constant average. A Wald test (and, separately, a likelihood comparison) asks whether the intervals differ. "Year 1" is the first calendar year after removal (removal dates are known only to the year).

In [ ]:
# The formulas exactly as fitted (recorded in the result files).
f = primary.dropna(subset=["formula"]) if "formula" in primary else pd.DataFrame()
for spec in ("before_after_distance", "baci_distance"):
    row = f[f["spec"] == spec].head(1)
    if len(row):
        print(f"{DESIGN[spec]} distance model:  {row['formula'].iloc[0]}")
settings_path = MODEL_DIR / "settings.json"
if settings_path.exists():
    import json
    for model, formula in json.loads(settings_path.read_text()).get("formulas", {}).items():
        print(f"{model} (10 km mixed model):  {formula}")
print("\nRandom effects chosen in the primary distance models:")
re_cols = [c for c in ("series", "spec", "dams", "re_structure") if c in primary]
display(primary[primary["series"].isin(SERIES) & primary["spec"].isin(DESIGN)][re_cols]
        .assign(design=lambda t: t["spec"].map(DESIGN)).drop(columns="spec")
        .pivot_table(index="series", columns="design", values="re_structure", aggfunc="first")
        .reindex([s for s in SERIES if s in set(primary["series"])]).rename(index=NAMES))

## 3. Key findings

The table gives the primary result for each measure and design: the **distance model's effect at 5 km downstream**, with its 95% interval, the number of dams, the verdict, and how the estimate compares with the practical-significance threshold (*share of threshold*: 1.0 = exactly at the threshold). The evidence tier combines the verdict with the robustness checks of Section 5.

In [ ]:
# Evidence tiers: an interpretive judgment combining the verdicts below with Section 5's checks.
TIER = {
    ("BA", "water_temp"): "strongest", ("BA", "dissolved_oxygen"): "strongest",
    ("BACI", "ph"): "strongest",
    ("BA", "orthophosphate (Dissolved)"): "supporting", ("BACI", "water_temp"): "supporting (distance pattern)",
    ("BACI", "dissolved_oxygen"): "suggestive (detected only under 3 / 1)",
    ("BACI", "turbidity"): "exploratory (small sample)",
    ("BA", "ph"): "detected but below threshold",
}

pm = primary[primary["series"].isin(SERIES) & primary["spec"].isin(DESIGN)].copy()
pm["design"] = pm["spec"].map(DESIGN)
rows = []
for _, r in pm.iterrows():
    key = (r["design"], r["series"])
    base = {"design": r["design"], "measure": NAMES[r["series"]], "dams": int(r["dams"])}
    if str(r.get("error", "") or "") not in ("", "nan"):
        rows.append({**base, "effect at 5 km [95% CI]": "not estimable (< 10 dams)", "verdict": "",
                     "threshold": threshold_text(r["series"]), "share of threshold": np.nan,
                     "evidence": "too few dams"})
        continue
    log = bool(r["log_scale"])
    v = verdict(r["effect_at_reference"], r["effect_ci_low"], r["effect_ci_high"], r["series"])
    tier = TIER.get(key, {"no meaningful change": "no meaningful change",
                          "inconclusive": "inconclusive"}.get(v, v))
    rows.append({**base, "effect at 5 km [95% CI]": fmt(r["effect_at_reference"], r["effect_ci_low"],
                                                       r["effect_ci_high"], r["series"], log),
                 "verdict": v, "threshold": threshold_text(r["series"]),
                 "share of threshold": abs(r["effect_at_reference"]) / SESOI[r["series"]],
                 "evidence": tier})
summary = (pd.DataFrame(rows).assign(_order=lambda t: t["measure"].map({NAMES[s]: i for i, s in enumerate(SERIES)}))
           .sort_values(["design", "_order"], ascending=[True, True]).drop(columns="_order"))
summary["sample"] = np.where((summary["dams"] < 15) & summary["verdict"].ne(""), "exploratory (< 15 dams)", "")
display(summary.style.format({"share of threshold": "{:.2f}"}, na_rep="")
        .hide(axis="index").set_caption("Primary results: distance models, effect at 5 km (latest corrected run)"))

# Guard: a tiered finding whose verdict no longer matches the narrative is flagged, not silently shown.
expected = {("BA", "water_temp"): "detected", ("BA", "dissolved_oxygen"): "detected",
            ("BA", "orthophosphate (Dissolved)"): "detected", ("BACI", "ph"): "detected",
            ("BACI", "dissolved_oxygen"): "inconclusive"}
for (d, s), want in expected.items():
    got = summary.loc[(summary["design"] == d) & (summary["measure"] == NAMES[s]), "verdict"]
    if got.empty or got.iloc[0] != want:
        print(f"CHECK: {d} {NAMES[s]} verdict is {got.iloc[0] if len(got) else 'missing'}, the text assumes {want}")

In [ ]:
# Figure 1: effects at 5 km relative to each measure's threshold.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True)
for ax, design in zip(axes, ["BA", "BACI"]):
    sub = pm[(pm["design"] == design) & pm["error"].fillna("").eq("")].copy()
    sub["order"] = sub["series"].map(SERIES.index)
    sub = sub.sort_values("order", ascending=False)
    ax.axvspan(-1, 1, color="0.92", lw=0, label="within threshold")
    ax.axvline(0, color="black", lw=0.8)
    for i, (_, r) in enumerate(sub.iterrows()):
        t = SESOI[r["series"]]
        v = verdict(r["effect_at_reference"], r["effect_ci_low"], r["effect_ci_high"], r["series"])
        ax.plot([r["effect_ci_low"] / t, r["effect_ci_high"] / t], [i, i], color=COLORS[v], lw=2)
        ax.plot(r["effect_at_reference"] / t, i, "o", color=COLORS[v], ms=6)
        ax.annotate(f"{int(r['dams'])} dams", (max(r["effect_ci_high"] / t, 0) + 0.1, i), va="center",
                    fontsize=7, color="0.4")
    ax.set_yticks(range(len(sub)), [NAMES[s] for s in sub["series"]])
    ax.set_title({"BA": "Before-after (downstream)", "BACI": "BACI (downstream vs upstream controls)"}[design])
    ax.set_xlabel("effect at 5 km ÷ threshold  (±1 = threshold)")
handles = [plt.Line2D([], [], color=c, marker="o", lw=2, label=k) for k, c in COLORS.items()]
fig.legend(handles=handles, loc="lower center", ncol=3, frameon=False, fontsize=8)
fig.suptitle("Figure 1. Estimated change after removal at 5 km, scaled by the analyst-selected threshold",
             fontsize=10)
fig.tight_layout(rect=(0, 0.06, 1, 0.95))
plt.show()

**Reading Figure 1.** Dots are estimates and bars 95% intervals, divided by each measure's threshold so measures with different units sit on one axis; the grey band is "within the threshold". Red = detected, green = no meaningful change, grey = inconclusive.

**Before-after (downstream), the larger samples:**
- **Water temperature decreases** (≈ −0.17 °C at 5 km, ≈ 270 dams) and **dissolved oxygen increases** (≈ +0.18 mg/L, ≈ 270 dams). Both are clearly detected and agree with the original 10 km mixed models (≈ −0.18 °C and +0.21 mg/L). Both are **smaller than the analyst-selected thresholds** (0.5 °C, 0.5 mg/L): detectable, but modest relative to the size of change we pre-defined as important.
- **Orthophosphate decreases** (≈ −17% at 5 km; ≈ −12% in the 10 km mixed model; fewer dams, ≈ 80). Detected, but more sensitive to modeling choices (Section 5).
- **pH**: a very small increase (≈ +0.03) is detected but lies well inside its threshold (0.1): **detectable but not practically meaningful** by our definition.
- **No meaningful change** (whole interval inside the threshold): total phosphorus, conductance, TDS, TSS and turbidity. This is evidence *of* no important change, not merely a lack of evidence.
- **Inconclusive**: suspended sediment (SSC, ≈ 30 dams), where the data can't tell.

**BACI (downstream vs upstream controls), the stronger design but ≈ 12–30 dams:**
- **pH increases** (≈ +0.11 at 5 km, ≈ 28 dams): the strongest comparative finding, at about the size of its threshold. The *detected* verdict doesn't depend on the threshold, but the size comparison does, and the pH threshold (±0.1) is the least supported: it equals the accuracy of a single pH reading. At ±0.2 the same effect is about half the threshold, so report the size in pH units rather than "meaningful".
- **Water temperature**: negligible effect at 5 km, but the effect **varies with distance** (Section 4).
- **Dissolved oxygen**: positive (≈ +0.3 mg/L) but **inconclusive** under the primary rule.
- **Turbidity**: positive and nominally detected (≈ +57%), but on **14 dams** (exploratory) and not robust (Section 5): treat as exploratory only.
- **Not estimable** (fewer than 10 dams with upstream controls): orthophosphate, TDS, TSS, SSC. Total phosphorus and conductance are estimable but inconclusive.

In [ ]:
# BACI turbidity: does the nominal detection survive leaving each dam out, and the dam bootstrap?
def influence_row(series):
    out = {}
    l = lodo[(lodo["series"] == series) & (lodo["spec"] == "baci_distance")] if len(lodo) else pd.DataFrame()
    b = boot[(boot["series"] == series) & (boot["spec"] == "baci_distance")] if len(boot) else pd.DataFrame()
    if len(l):
        r = l.iloc[0]
        out.update({"dams": r.get("dams"), "effect range leaving one dam out": f"{r['lodo_effect_min']:+.3f} to {r['lodo_effect_max']:+.3f}",
                    "refits where 'CI excludes 0' flips": r.get("effect_conclusion_changes"),
                    "influential dams": r.get("influential_dam_ids") if isinstance(r.get("influential_dam_ids"), str) and r.get("influential_dam_ids") else "none"})
    if len(b) and "effect_at_reference_boot_low" in b:
        r = b.iloc[0]
        out["bootstrap 95% interval"] = f"[{r['effect_at_reference_boot_low']:+.3f}, {r['effect_at_reference_boot_high']:+.3f}]"
    return out

infl = pd.DataFrame({NAMES[s]: influence_row(s) for s in ("turbidity", "ph", "water_temp", "dissolved_oxygen")}).T
if infl.empty:
    print("Influence results (stage4/influence/) not found.")
else:
    print("Influence checks for the BACI distance models (log10 units for turbidity):")
    display(infl)

**Decision on BACI turbidity.** It stays in the **exploratory** tier regardless of the influence check above: 14 dams (below the 15-dam exploratory line), weaker under the 3 / 1 rule, and not estimable with confirmed controls. If the leave-one-dam-out row shows refits where the conclusion flips, that is one more reason not to emphasize it.

## 4. How responses vary with distance and time

In [ ]:
# Figure 2: effect vs distance for the findings where distance matters.
panels = [("before_after_distance", "water_temp"), ("before_after_distance", "dissolved_oxygen"),
          ("baci_distance", "water_temp"), ("baci_distance", "ph")]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, (spec, series) in zip(axes, panels):
    a = primary_at[(primary_at["spec"] == spec) & (primary_at["series"] == series)
                   & primary_at["distance_km"].notna()].sort_values("distance_km") if len(primary_at) else pd.DataFrame()
    t = SESOI[series]
    ax.axhspan(-t, t, color="0.92", lw=0)
    ax.axhline(0, color="black", lw=0.8)
    if len(a):
        ax.fill_between(a["distance_km"], a["ci_low"], a["ci_high"], color="#2980b9", alpha=0.2, lw=0)
        ax.plot(a["distance_km"], a["estimate"], "o-", color="#2980b9", ms=4)
    ax.set_xscale("log")
    ax.set_xticks([1, 5, 10, 20, 30], ["1", "5", "10", "20", "30"])
    ax.set_xlabel("km downstream")
    ax.set_ylabel({"water_temp": "change (°C)", "dissolved_oxygen": "change (mg/L)", "ph": "change (pH units)"}[series])
    ax.set_title(f"{DESIGN[spec]}: {NAMES[series]}")
fig.suptitle("Figure 2. Estimated change after removal by distance downstream (band = 95% CI; grey = within threshold)",
             fontsize=10)
fig.tight_layout(rect=(0, 0, 1, 0.92))
plt.show()

slopes = pm[pm["error"].fillna("").eq("") & pm["series"].isin(["water_temp", "dissolved_oxygen", "ph", "orthophosphate (Dissolved)"])]
slopes = slopes.assign(measure=slopes["series"].map(NAMES),
                       slope=[f"{e:+.3f} [{l:+.3f}, {h:+.3f}]" for e, l, h in zip(slopes["modifier_slope"], slopes["modifier_ci_low"], slopes["modifier_ci_high"])],
                       detected=(slopes["modifier_ci_low"] > 0) | (slopes["modifier_ci_high"] < 0))
display(slopes[["design", "measure", "dams", "slope", "detected"]].sort_values(["design", "measure"])
        .style.hide(axis="index").set_caption("Distance slope: change in the effect per unit of ln(1 + km), model units"))

**Distance.**
- **BA temperature and DO effects fade downstream**: both slopes are detected and opposite in sign to the effect, so the cooling and the oxygen gain are largest near the dam and shrink farther downstream (Figure 2 shows where the interval reaches zero).
- **BACI temperature depends on distance**: the slope is detected, so relative to upstream controls the estimate runs from cooler near the dam toward warmer farther away; its effect *at 5 km* is negligible. The ends of the curve rest on few stations and have wide intervals, so read it as "the response is not uniform along the river" rather than as precise values.
- **BACI pH is roughly flat** with distance (slope not detected): the increase appears similar along the first 30 km.
- **Orthophosphate (BA)** shows no clear distance pattern.

In [ ]:
# Does the response change over the five years after removal? (time models, primary rule)
def conclusion(p):
    return "" if pd.isna(p) else ("changes over time" if p < 0.05 else "no evidence of change")

if time_cmp.empty:
    print("Time-model outputs not found.")
else:
    t = time_cmp[time_cmp["series"].isin(["water_temp", "dissolved_oxygen", "ph", "orthophosphate (Dissolved)", "turbidity"])]
    t = t[["series", "design", "dams", "time_change_p"]].copy()
    if len(time_slopes):
        t = t.merge(time_slopes[["series", "design", "lrt_p", "delta_aic_time_minus_binary"]], on=["series", "design"], how="left")
    t["Wald test"] = t["time_change_p"].map(conclusion)
    if "lrt_p" in t:
        t["likelihood test (dam-specific effects)"] = t["lrt_p"].map(conclusion)
    t["design"] = t["design"].map({"baci": "BACI", "before_after": "BA"})
    t["measure"] = t["series"].map(NAMES)
    cols = ["design", "measure", "dams", "time_change_p", "Wald test"] + \
           [c for c in ("lrt_p", "likelihood test (dam-specific effects)") if c in t]
    display(t[cols].sort_values(["design", "measure"]).style.format({"time_change_p": "{:.3f}", "lrt_p": "{:.4f}"}, na_rep="")
            .hide(axis="index").set_caption("Is one constant effect enough, or does it change across years 1, 2-3 and 4-5?"))

**Time.**
- **Stable over the first five years: BA dissolved oxygen and BA temperature.** One constant effect describes them well (both tests agree).
- **Changing over time: BACI temperature** (both tests, both eligibility rules), and **BA pH, turbidity and orthophosphate** (the orthophosphate decrease is clearest in years 4–5).
- **Inconsistent: BACI pH.** The two tests disagree and the answer flips under the looser eligibility rule, so we make no claim about its time pattern; its overall increase is robust (Section 5).
- Later intervals rest on fewer dams, so time patterns are less certain than the overall effects.

### Why do responses differ between sites?

Two questions, both answered from existing outputs (nothing is refit):

1. **How much do dams differ?** The distance models can give each dam its own effect (a *random slope*), kept only where a likelihood-ratio test supports it (`RE = auto`). Its standard deviation (SD) measures how much the true effect varies from dam to dam around the average. If the test didn't support it, the data can't tell the dams' effects apart from one shared effect, given the noise.
2. **Which dam and watershed characteristics go with bigger or smaller effects?** The *modifier screen* refits each model with one characteristic at a time (dam height, reservoir size, land cover, …; distance and dilution held fixed) and reports how the effect changes **per SD of that characteristic**. Because many characteristics × measures are tested, each result has a **q-value** (Benjamini–Hochberg false-discovery rate across the screen). Full detail: `notebooks/analysis/time_and_modifier_results.ipynb`, sections 6–12.

In [ ]:
# (1) How much do dams differ? Random-slope SD of the effect at 5 km, primary distance models.
from scipy import stats

SLOPE_TERM = {"before_after_distance": "post", "baci_distance": "post_downstream"}
Z80 = stats.norm.ppf(0.9)   # middle 80% of a normal distribution of dam effects

def natural_text(v, series, log):
    if pd.isna(v):
        return ""
    d = 0 if log else 2
    return f"{natural(v, log):+.{d}f}"

het_rows = []
for _, r in pm[pm["error"].fillna("").eq("")].iterrows():
    log, est, thr = bool(r["log_scale"]), r["effect_at_reference"], SESOI[r["series"]]
    var = r.get(f"var_dam_{SLOPE_TERM[r['spec']]}", np.nan)
    sd = np.sqrt(var) if pd.notna(var) and var > 0 else np.nan
    u = unit_of(r["series"], log)
    het_rows.append({
        "design": r["design"], "measure": NAMES[r["series"]], "dams": int(r["dams"]),
        "average effect": f"{natural_text(est, r['series'], log)} {u}".strip(),
        "dam-to-dam SD ÷ threshold": round(sd / thr, 2) if pd.notna(sd) else np.nan,
        "middle 80% of dams (model-based)": (f"{natural_text(est - Z80 * sd, r['series'], log)} to "
                                             f"{natural_text(est + Z80 * sd, r['series'], log)} {u}".strip()
                                             if pd.notna(sd) else ""),
        "dams with the opposite sign (model-based)": (f"{stats.norm.cdf(-abs(est) / sd):.0%}" if pd.notna(sd) else ""),
        "dam-specific effects": ("supported (kept)" if pd.notna(sd) else
                                 "not supported: one shared effect" if "intercept" in str(r.get("re_choice", ""))
                                 else "not available"),
    })
het = (pd.DataFrame(het_rows).assign(_o=lambda t: t["measure"].map({NAMES[s]: i for i, s in enumerate(SERIES)}))
       .sort_values(["design", "_o"]).drop(columns="_o"))
display(het.style.hide(axis="index").format({"dam-to-dam SD ÷ threshold": "{:.2f}"}, na_rep="")
        .set_caption("How much the effect at 5 km varies between dams (primary distance models)"))
kept = het[het["dam-specific effects"].str.startswith("supported")]
wide = kept[kept["dam-to-dam SD ÷ threshold"] > 1]
print(f"Dam-specific effects supported for {len(kept)} of {len(het)} models. "
      f"Dam-to-dam SD above the threshold: {', '.join(wide['design'] + ' ' + wide['measure']) or 'none'}.")

In [ ]:
# (2) Which characteristics go with bigger or smaller effects? The modifier screen (distance and dilution held fixed).
from analysis import distance_models as dm

screen = load("stage5", "modifiers", "modifier_screen.csv")
if screen.empty:
    print("Modifier screen not found (stage5/modifiers__*/modifier_screen.csv).")
else:
    src = run_dir("stage5", "modifiers") / "modifier_screen.csv"
    written = datetime.fromtimestamp(src.stat().st_mtime)
    print(f"Source: {src.relative_to(MODEL_DIR)}, written {written:%Y-%m-%d %H:%M}"
          + ("" if written >= CORRECTED_ON else "  WARNING: predates the 8 Nov 2026 corrections"))
    s = screen[screen["series"].isin(SERIES) & screen["error"].fillna("").eq("")].copy()
    if "adjusted_for_distance_dilution" in s:
        s = s[s["adjusted_for_distance_dilution"].astype(str).str.lower().eq("true")]
    s["design"] = s["design"].map({"baci": "BACI", "before_after": "BA"}).fillna(s["design"])
    thr = s["series"].map(SESOI).astype(float)
    for c in ("modifier_slope", "modifier_ci_low", "modifier_ci_high"):
        s[c + "_t"] = s[c] / thr
    s["ci_excludes_0"] = (s["modifier_ci_low"] > 0) | (s["modifier_ci_high"] < 0)
    k80 = stats.t.ppf(0.975, (s["dams"] - 1).clip(lower=1)) + stats.t.ppf(dm.POWER, (s["dams"] - 1).clip(lower=1))
    s["power"] = dm.power(thr.to_numpy(), (s["mde80_modifier"] / k80).to_numpy(), s["dams"].to_numpy(float))

    overview = (s.groupby("design").agg(tests=("characteristic", "size"),
                                        ci_excludes_0=("ci_excludes_0", "sum"),
                                        q_below_0_1=("q_value", lambda q: int((q < 0.1).sum())),
                                        median_power=("power", "median"))
                .assign(expected_by_chance=lambda t: (0.05 * t["tests"]).round(1)))
    overview = overview[["tests", "ci_excludes_0", "expected_by_chance", "q_below_0_1", "median_power"]]
    display(overview.style.format({"median_power": "{:.2f}", "expected_by_chance": "{:.1f}"})
            .set_caption("Characteristic screen: how many associations, versus chance (power: one threshold per SD)"))

    leads = s[s["ci_excludes_0"]].sort_values("q_value")
    if leads.empty:
        print("No characteristic has a 95% interval excluding 0.")
    else:
        leads = pd.DataFrame({
            "design": leads["design"], "measure": leads["series"].map(NAMES), "characteristic": leads["characteristic"],
            "dams": leads["dams"].astype(int),
            "change in effect per SD ÷ threshold [95% CI]": [f"{e:+.2f} [{l:+.2f}, {h:+.2f}]" for e, l, h in
                                                              zip(leads["modifier_slope_t"], leads["modifier_ci_low_t"], leads["modifier_ci_high_t"])],
            "q-value": leads["q_value"].round(3),
            "standing": np.where(leads["q_value"] < 0.1, "survives multiple testing (q < 0.1)", "lead only"),
            "per SD of": leads.get("modifier_scale", pd.Series("", index=leads.index)).fillna(""),
        })
        display(leads.style.hide(axis="index").format({"q-value": "{:.3f}"}).set_caption(
            "Characteristics whose 95% interval excludes 0 (+ = larger effect at dams higher in the characteristic)"))

**How to read this.**
- **Dam-to-dam SD ÷ threshold** above 1 means individual dams' true effects commonly differ from the average by more than the threshold, so the average hides real site-to-site variation. The "middle 80%" and "opposite sign" columns assume dam effects are normally distributed around the average. They describe the model's estimate of the spread, not observed dam-by-dam results, and they ignore the uncertainty in the average itself.
- **"One shared effect"** doesn't prove that dams respond identically. It means the data can't separate dam-specific effects from noise.
- **The characteristic screen is exploratory.** A characteristic whose interval excludes 0 but with q ≥ 0.1 is a *lead*. About 5% of tests are expected to exclude 0 by chance (the *expected by chance* column). Only *q < 0.1* results survive the multiple testing.
- **Low power is the main limitation:** most screens can only detect a large modification (the *median power* column, for a change of one threshold per SD). **No leads does not mean characteristics don't matter**, only that this sample can't detect it.
- These are **associations across dams**, not causes. Many characteristics travel together (e.g. land cover, region and reservoir size; time and modifier notebook section 7), so a lead may stand in for a correlated factor.

## 5. How robust are the findings?

Four pre-specified checks, each compared with the primary result using fixed labels: **robust** (same verdict and direction), **sharper** (inconclusive → decided), **weaker** (decided → inconclusive), **conflicting** (opposite verdicts or direction with a clear signal), **not comparable** (fewer than 10 dams in either version).

1. **Eligibility rule.** Primary: a station needs ≥ 3 samples in ≥ 2 different years on each side of the removal (*3 / 2*). Sensitivity: ≥ 3 samples in ≥ 1 year (*3 / 1*), which adds dams, and *3 / 1 with ≥ 2 months* (drops single-month sides).
2. **Upstream controls known to be outside the former reservoir.** A station inside the drained reservoir is itself changed by the removal, so it is not a valid control. The primary analysis counts stations of *unknown* status as controls; this check keeps only confirmed ones.
3. **Influence:** leave-one-dam-out and a dam bootstrap (BACI distance models, Section 3).
4. **Model choice:** the original 10 km mixed models vs the 30 km distance models.

In [ ]:
# Robustness of the main findings, compared with the pre-specified labels.
def labels(main, other, keys, est, lo, hi):
    if main.empty or other.empty:
        return pd.Series(dtype=str)
    t = sc.compare(main, other, keys, est, lo, hi, sesoi=SESOI)
    return t.set_index(keys)["label"]

dist = lambda f: f[f["spec"].isin(DESIGN)] if len(f) else f
mixed_baci = lambda f: f[f["model"] == "baci"] if len(f) else f
FINDINGS = [("BA", "water_temp"), ("BA", "dissolved_oxygen"), ("BA", "orthophosphate (Dissolved)"),
            ("BA", "ph"), ("BACI", "ph"), ("BACI", "water_temp"), ("BACI", "dissolved_oxygen"), ("BACI", "turbidity")]
SPEC = {"BA": "before_after_distance", "BACI": "baci_distance"}
MIXED = {"BA": "before_after", "BACI": "baci"}

cols = {}
for label, (s_dist, s_mixed) in sens.items():
    cols[f"{label} (5 km)"] = labels(dist(primary), dist(s_dist), ["series", "spec"], "effect_at_reference", "effect_ci_low", "effect_ci_high")
cols["known controls (5 km)"] = labels(dist(primary), dist(known), ["series", "spec"], "effect_at_reference", "effect_ci_low", "effect_ci_high")
mcols = {"3 / 1 (10 km mixed)": labels(mixed, sens["3 / 1"][1], ["series", "model"], "estimate", "ci_low", "ci_high"),
         "known controls (10 km mixed)": labels(mixed_baci(mixed), mixed_baci(known_mixed), ["series", "model"], "estimate", "ci_low", "ci_high")}

rows = []
for design, series in FINDINGS:
    r = {"finding": f"{design}: {NAMES[series]}"}
    m = primary[(primary["series"] == series) & (primary["spec"] == SPEC[design])]
    if len(m):
        r["primary verdict"] = verdict(m["effect_at_reference"].iloc[0], m["effect_ci_low"].iloc[0], m["effect_ci_high"].iloc[0], series)
    mx = mixed[(mixed["series"] == series) & (mixed["model"] == MIXED[design])] if len(mixed) else pd.DataFrame()
    if len(mx) and pd.notna(mx["estimate"].iloc[0]):
        r["10 km mixed model"] = verdict(mx["estimate"].iloc[0], mx["ci_low"].iloc[0], mx["ci_high"].iloc[0], series)
    for name, lab in cols.items():
        if name.startswith("known") and design == "BA":
            r[name] = "n/a (no controls)"
        else:
            r[name] = lab.get((series, SPEC[design]), "")
    for name, lab in mcols.items():
        if name.startswith("known") and design == "BA":
            r[name] = "n/a (no controls)"
        else:
            r[name] = lab.get((series, MIXED[design]), "")
    rows.append(r)
robust = pd.DataFrame(rows).set_index("finding")
display(robust.style.set_caption("Robustness of the main findings (labels vs the primary result)"))

In [ ]:
# How much of BACI rests on unconfirmed controls, and are the dams that lose them different?
used_main = read_csv(run_dir("stage3", "primary") / "dams_used.csv") if run_dir("stage3", "primary") else pd.DataFrame()
kc_dir = run_dir("sensitivity_known_controls", "known_controls")
used_kc = read_csv(kc_dir / "dams_used.csv") if kc_dir else pd.DataFrame()
if len(used_main) and len(used_kc):
    p_ids = set(used_main.loc[used_main["spec"] == "baci_distance", "AR_ID"].astype(str))
    kept = set(used_kc.loc[used_kc["spec"] == "baci_distance", "AR_ID"].astype(str)) & p_ids
    print(f"BACI dams in the primary models: {len(p_ids)}; with confirmed controls: {len(kept)}; lose all controls: {len(p_ids) - len(kept)}")
    try:
        from etl.common.config import load_config
        from etl.common.warehouse import analysis_path, warehouse_dir
        wh = Path(os.environ.get("DAM_REMOVAL_WAREHOUSE_DIR") or warehouse_dir(load_config(CASE_STUDY, project_root=DATA_ROOT)))
        need = ["AR_ID", "station_id", "side"] + sorted({c for c, _, _ in sc.CHARACTERISTICS.values()})
        chars = sc.dam_characteristics(pd.read_parquet(analysis_path(wh), columns=need))
        d = sc.compare_dams(chars, kept, p_ids - kept)
        d = d[d["systematic"]][["characteristic", "primary_mean", "added_mean", "smd_added_minus_primary"]]
        d.columns = ["characteristic", "mean: keep controls", "mean: lose controls", "SMD (lose − keep)"]
        display(d.style.format({"mean: keep controls": "{:.3g}", "mean: lose controls": "{:.3g}", "SMD (lose − keep)": "{:+.2f}"})
                .hide(axis="index").set_caption("Characteristics that differ systematically (|SMD| ≥ 0.25)"))
    except Exception as exc:
        print(f"Dam characteristics need the warehouse analysis table: {exc}")
else:
    print("Known-controls outputs not found.")

**What changes the interpretation**

- **BA temperature and DO are stable** across the eligibility rules and both model types: the most solid results.
- **BACI pH is comparatively robust**: it keeps its direction and size under the looser rule *and* with only confirmed controls (on about half the dams).
- **BACI DO is sensitive to the eligibility rule**: inconclusive under the primary rule, detected only under 3 / 1, so it is reported as suggestive, not as a finding.
- **Orthophosphate (BA)** holds in the distance model but weakens under 3 / 1 in the 10 km mixed model: moderately supported.
- **Confirmed controls halve the BACI sample**, and the dams that lose their controls differ systematically (more agricultural, less forested, removed earlier; table above). BACI results therefore partly reflect *which* dams have confirmable controls, not just how many.
- **"Conflicting" BA pH** is a zero-line effect: a tiny estimate whose interval only just excludes zero under one rule and not the other. It doesn't change any conclusion.

**Data-quality corrections (8 Nov 2026).** A QA review found reporting errors in the public data and fixed them for every analysis: conductance and TDS values reported in the wrong unit (e.g. mS/cm labelled µS/cm at one dam's stations), standard non-detect codes that weren't recognized, and zeros recorded for below-detection readings. Effect on the results (before vs after correction; pre-correction values are from the archived 7 Nov notebook exports, since the refit replaced those outputs):

| Result | Before correction | After |
|---|---|---|
| BA TDS (10 km mixed) | +5.8%, inconclusive | −1.9%, no meaningful change (a spurious increase removed) |
| BA turbidity (10 km mixed) | +4.0% | −1.6% (no meaningful change either way) |
| BA SSC (10 km mixed, 7 dams) | −42% | −33% |
| BACI turbidity (5 km) | +0.23 log10, inconclusive | +0.20 log10, detected (exploratory) |
| BA conductance, 3 / 1 vs primary (CI width ratio) | 4.1× (one dam's unit error) | 1.3× |
| Temperature, DO, pH, orthophosphate | — | unchanged |

## 6. What can we conclude?

**Synthesis.** Across ≈ 270 removals with downstream monitoring, removal is **associated with slightly cooler and better-oxygenated water downstream**, most noticeable near the dam and fading within a few tens of kilometers, and stable over the first five years. Orthophosphate tends to decrease. Where upstream controls exist (≈ 30 dams), the clearest comparative signal is a **small rise in pH**, robust to the checks we ran. Most other measures show **no meaningful change**, and suspended sediment and most BACI nutrient/solids results are **inconclusive or not estimable** for lack of data.

**Limitations**
- **Observational data:** associations, not proof that removal caused the changes; BACI reduces but doesn't remove this concern.
- **Small BACI samples** (≈ 12–30 dams), several of them dependent on upstream controls whose former-reservoir status is unknown.
- **Former reservoir areas are not characterized** and are excluded; they may be where change is largest (future work).
- **Uneven monitoring:** stations, sampling frequency, seasons, methods and organizations vary across sites and over time.
- **Site-specific confounding**, e.g. CO-025, where pre-removal sediment was likely raised by a 2012 wildfire and 2013 floods.
- **Analyst-selected thresholds:** the "no meaningful change" verdicts and statements about magnitude depend on them (a 0.5×–2× sensitivity exists); *detected* verdicts do not. They are single national values for changes in typical levels, while ecological importance depends on baseline, region and duration; published criteria are mostly absolute levels, not sizes of change. Literature support is weakest for pH (threshold = instrument accuracy), TDS, orthophosphate and TSS (`docs/practical_significance_thresholds.md`).
- **Many outcomes and models** (10 measures × 2 designs, plus distance and time) with **no overall multiple-testing correction**; isolated detections should be read cautiously.

**Overall takeaway.** *Dam removal is associated with small, localized improvements in downstream temperature and dissolved oxygen that persist over the first five years, and with a small pH increase where upstream controls allow a comparison; most other water-quality measures show no meaningful change. The evidence is strongest for the large before-after samples and limited for the comparative (BACI) design by the scarcity of upstream controls.*

### What could the data detect? (post-hoc design sensitivity)

*Not a pre-specified or prospective power analysis: added 9 Nov 2026 after the results were seen, from the standard errors already in the result files. It changes no estimate, verdict, threshold or conclusion above.*

Every model can only detect effects above a certain size. The **minimum detectable effect (MDE)** is the true change a model would detect 80% of the time; **power at threshold** is the chance it would detect a change exactly as large as the analyst-selected, provisional "too small to matter" threshold (never the observed estimate). Both use dams − 1 degrees of freedom, as the 95% intervals do, which for mixed models is an approximation. For log-scale measures the MDE and threshold are compared on the log scale and shown as % changes.

**Why it matters for the conclusions:** where power is high, *inconclusive* or *no meaningful change* verdicts are informative; where it is low (listed under the table), they mean the data were too thin to tell, which is part of the "limited BACI evidence" limitation above. The last column is a rough projection under strong assumptions, not a sample-size requirement.

In [ ]:
# Design sensitivity of the primary results (distance models, effect at 5 km). Model scale throughout.
from scipy import stats
from analysis import distance_models as dm

ok = pm[pm["error"].fillna("").eq("") & pm["mde80_effect"].notna()].copy() if "mde80_effect" in pm else pd.DataFrame()
if ok.empty:
    print("No MDE in the primary results (mde80_effect missing).")
else:
    thr = ok["series"].map(SESOI).astype(float)
    log = ok["log_scale"].astype(str).str.lower().eq("true")
    se = ok["effect_se"].astype(float) if "effect_se" in ok else ok["mde80_effect"] / (
        stats.t.ppf(0.975, ok["dams"] - 1) + stats.t.ppf(dm.POWER, ok["dams"] - 1))
    ok["power"] = dm.power(thr.to_numpy(), se.to_numpy(), ok["dams"].to_numpy(float))
    ok["need"] = [dm.dams_needed(m, t, d) for m, t, d in zip(ok["mde80_effect"], thr, ok["dams"])]
    def mde_text(m, l, s):
        return f"+{(10 ** m - 1) * 100:.0f}% / −{(1 - 10 ** -m) * 100:.0f}%" if l else f"±{m:.2g} {unit_of(s, False)}".strip()
    power_table = pd.DataFrame({
        "design": ok["design"], "measure": ok["series"].map(NAMES), "dams": ok["dams"].astype(int),
        "verdict": [verdict(e, lo, hi, s) for e, lo, hi, s in
                    zip(ok["effect_at_reference"], ok["effect_ci_low"], ok["effect_ci_high"], ok["series"])],
        "threshold": ok["series"].map(threshold_text),
        "detectable change (80%)": [mde_text(m, l, s) for m, l, s in zip(ok["mde80_effect"], log, ok["series"])],
        "power at threshold": ok["power"].round(2),
        "dams to detect the threshold (rough)": ["already enough" if m <= t else ("> 10,000" if np.isinf(n) else f"~{n:,.0f}")
                                                  for m, t, n in zip(ok["mde80_effect"], thr, ok["need"])],
    }).sort_values(["design", "power at threshold"], ascending=[True, False])
    display(power_table.style.hide(axis="index").format({"power at threshold": "{:.2f}"})
            .set_caption("What each primary model could detect (post-hoc; power at the provisional threshold)"))
    high = power_table[power_table["power at threshold"] >= 0.8]
    low = power_table[power_table["power at threshold"] < 0.5]
    print(f"≥ 80% power at threshold: {len(high)} of {len(power_table)} "
          f"({', '.join(high['design'] + ' ' + high['measure']) or 'none'}).")
    print(f"< 50% power at threshold: {len(low)} "
          f"({', '.join(low['design'] + ' ' + low['measure']) or 'none'}).")

### Is "inconclusive" always a lack of data?

**"No meaningful change" and "inconclusive" are different categories by construction** (`stage4_report.classify`):
- **No meaningful change:** the whole 95% interval lies inside ± threshold. The data are precise enough to rule out a threshold-sized change in either direction. This is a positive finding of a small effect.
- **Inconclusive:** the interval includes 0 *and* reaches past the threshold. The data can't separate a meaningful change from none.

An inconclusive result can have two causes, which the table below separates using the same post-hoc quantities as above:
- **Too imprecise** (MDE80 > threshold): even a threshold-sized change would usually be missed. This is the "insufficient data" case: too few dams, sparse sampling, or dams that disagree.
- **Precise, but unresolved** (MDE80 ≤ threshold): the model could detect a threshold-sized change, but the estimate lies between 0 and the threshold, so a moderate effect can be neither confirmed nor ruled out. Here more data would also help, but the result is not simply "no data".

In [ ]:
# Every primary result that is not "detected": why it is "no meaningful change" or "inconclusive".
nd = pm[pm["error"].fillna("").eq("") & pm["mde80_effect"].notna()].copy() if "mde80_effect" in pm else pd.DataFrame()
if nd.empty:
    print("No MDE in the primary results (mde80_effect missing).")
else:
    thr = nd["series"].map(SESOI).astype(float)
    nd["verdict"] = [verdict(e, lo, hi, s) for e, lo, hi, s in
                     zip(nd["effect_at_reference"], nd["effect_ci_low"], nd["effect_ci_high"], nd["series"])]
    nd["est_t"] = nd["effect_at_reference"].abs() / thr
    nd["half_t"] = (nd["effect_ci_high"] - nd["effect_ci_low"]) / 2 / thr
    nd["mde_t"] = nd["mde80_effect"] / thr
    nd = nd[nd["verdict"].ne("detected")]

    def reason(r):
        if r["verdict"] == "no meaningful change":
            return "precise and small: whole interval inside ± threshold"
        if r["mde_t"] > 1:
            return "too imprecise: a threshold-sized change would usually be missed"
        return "precise, but unresolved: estimate between 0 and the threshold"

    nd["why"] = nd.apply(reason, axis=1)
    why = pd.DataFrame({"design": nd["design"], "measure": nd["series"].map(NAMES), "dams": nd["dams"].astype(int),
                        "verdict": nd["verdict"], "|estimate| ÷ threshold": nd["est_t"].round(2),
                        "95% CI half-width ÷ threshold": nd["half_t"].round(2), "MDE80 ÷ threshold": nd["mde_t"].round(2),
                        "why": nd["why"]}).sort_values(["verdict", "design", "MDE80 ÷ threshold"])
    display(why.style.hide(axis="index").format({c: "{:.2f}" for c in ["|estimate| ÷ threshold",
                                                                        "95% CI half-width ÷ threshold", "MDE80 ÷ threshold"]})
            .set_caption("Results that are not 'detected': no meaningful change vs inconclusive, and why"))
    inc = why[why["verdict"] == "inconclusive"]
    imprecise = inc[inc["why"].str.startswith("too imprecise")]
    unresolved = inc[inc["why"].str.startswith("precise, but")]
    print(f"Inconclusive: {len(inc)}. Too imprecise: {len(imprecise)} "
          f"({', '.join(imprecise['design'] + ' ' + imprecise['measure']) or 'none'}). "
          f"Precise but unresolved: {len(unresolved)} ({', '.join(unresolved['design'] + ' ' + unresolved['measure']) or 'none'}).")
    nmc = why[why["verdict"] == "no meaningful change"]
    print(f"Check: every 'no meaningful change' has a CI half-width below the threshold: {bool((nmc['95% CI half-width ÷ threshold'] < 1).all())}.")

**Reading it.** "No meaningful change" rows are informative and belong with the findings. "Too imprecise" rows are honest unknowns that more dams or more sampling could resolve. "Precise but unresolved" rows deserve a sentence of their own: a moderate effect is plausible but unconfirmed. The MDE uses dams − 1 degrees of freedom (an approximation for mixed models), and the split is a post-hoc reading of the existing intervals. It changes no verdict.

## Summary

**Key messages for the presentation**
1. **Downstream water gets slightly cooler and more oxygenated after removal** (≈ −0.17 °C, ≈ +0.2 mg/L near the dam), fading downstream and stable over five years: detectable, but below our analyst-selected importance thresholds.
2. **The strongest comparative result is a small pH increase** (BACI, ≈ +0.11 at 5 km), robust to the eligibility rule and to confirmed controls.
3. **Most measures show no meaningful change**, which is informative in itself; sediment and most BACI nutrient results lack the data to say.
4. **The design matters:** BA has hundreds of dams but no comparison group; BACI is stronger but has ≈ 30 dams, half of which depend on unconfirmed controls.
5. **Data quality work changed results:** fixing reporting errors removed a spurious TDS increase and made the sensitivity analyses consistent.

**Emphasize:** BA temperature and DO; BACI pH; the BA/BACI distinction; the robustness checks.

**Mention only as supporting or exploratory:** BA orthophosphate (supporting); the BACI temperature distance pattern (supporting); BACI DO (suggestive, eligibility-sensitive); BACI turbidity (exploratory, small sample); time patterns of pH, turbidity and orthophosphate.

**Avoid:** presenting the thresholds as ecological or regulatory standards; causal language ("removal caused"), calling sub-threshold effects "important" or "negligible" without naming the threshold, and presenting the former reservoir as studied.

In [ ]:
# Narrative check: the approximate numbers in the text vs the loaded results (flags drift; changes nothing).
claims = [("before_after_distance", "water_temp", -0.17, 0.03), ("before_after_distance", "dissolved_oxygen", 0.18, 0.04),
          ("before_after_distance", "orthophosphate (Dissolved)", np.log10(0.83), 0.02),
          ("before_after_distance", "ph", 0.03, 0.02), ("baci_distance", "ph", 0.11, 0.03),
          ("baci_distance", "dissolved_oxygen", 0.33, 0.1), ("baci_distance", "turbidity", np.log10(1.57), 0.05)]
out = []
for spec, series, stated, tol in claims:
    m = primary[(primary["spec"] == spec) & (primary["series"] == series)]
    got = m["effect_at_reference"].iloc[0] if len(m) else np.nan
    out.append({"result": f"{DESIGN[spec]} {NAMES[series]}", "stated (model units)": stated, "loaded": got,
                "ok": bool(np.isfinite(got) and abs(got - stated) <= tol)})
check = pd.DataFrame(out)
display(check.style.format({"stated (model units)": "{:+.3f}", "loaded": "{:+.3f}"}).hide(axis="index"))
print("All narrative numbers match the loaded results." if check["ok"].all()
      else "Some narrative numbers differ from the loaded results: update the text before presenting.")